# Matrix calculus: gradient identities

_Linear Algebra_

**A handful of derivative rules for vectors and matrices that turn least squares into one line.**

You already know the gradient of a scalar function of a vector: stack the partial derivatives
        into a vector that points uphill (see fnd-gradient). Matrix calculus extends that
        idea to scalars built from matrices too.

       The key fact is about shape: the gradient of a scalar with respect to a variable has the
        same shape as that variable. If $f$ is a number and $x$ is a length-$n$ vector, then
        $\nabla_x f$ is a length-$n$ vector. If $f$ is a number and $A$ is an $m\times n$ matrix, then
        $\nabla_A f$ is an $m\times n$ matrix, and its entry $(i,j)$ is just $\partial f/\partial A_{ij}$ —
        the ordinary partial derivative with respect to that one entry.

---

This notebook is a practice scaffold for the **Matrix calculus: gradient identities** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — NumPy

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def numgrad_vec(f, x, h=1e-6):
    """Central finite-difference gradient of a scalar f(vector)."""
    g = np.zeros_like(x)
    for i in range(x.size):
        xp, xm = x.copy(), x.copy()
        xp[i] += h; xm[i] -= h
        g[i] = (f(xp) - f(xm)) / (2 * h)
    return g

def numgrad_mat(f, A, h=1e-6):
    """Central finite-difference gradient of a scalar f(matrix)."""
    G = np.zeros_like(A)
    for idx in np.ndindex(A.shape):
        Ap, Am = A.copy(), A.copy()
        Ap[idx] += h; Am[idx] -= h
        G[idx] = (f(Ap) - f(Am)) / (2 * h)
    return G

n = 5
b = rng.standard_normal(n); x = rng.standard_normal(n)
A = rng.standard_normal((n, n))

# 1) grad_x b^T x = b
err1 = np.max(np.abs(b - numgrad_vec(lambda v: b @ v, x)))
# 2) grad_x x^T A x = (A + A^T) x
err2 = np.max(np.abs((A + A.T) @ x - numgrad_vec(lambda v: v @ A @ v, x)))

m = 4
B = rng.standard_normal((m, m)); C = rng.standard_normal((m, m))
M = rng.standard_normal((m, m))
# 3) grad_A tr(A B) = B^T
err3 = np.max(np.abs(B.T - numgrad_mat(lambda Z: np.trace(Z @ B), M)))
# 4) grad_A tr(A B A^T C) = C A B + C^T A B^T
ana4 = C @ M @ B + C.T @ M @ B.T
err4 = np.max(np.abs(ana4 - numgrad_mat(lambda Z: np.trace(Z @ B @ Z.T @ C), M)))
# 5) grad_A |A| = |A| (A^-1)^T
ana5 = np.linalg.det(M) * np.linalg.inv(M).T
err5 = np.max(np.abs(ana5 - numgrad_mat(lambda Z: np.linalg.det(Z), M)))

for name, e in [("b^T x", err1), ("x^T A x", err2), ("tr(AB)", err3),
                ("tr(ABA^T C)", err4), ("|A|", err5)]:
    print(f"{name:14s} analytic vs numerical max abs error = {e:.3e}")

# Normal equations: minimizing ||Xw - y||^2 gives w = (X^T X)^-1 X^T y,
# i.e. the gradient 2 X^T (X w - y) is zero at the solution.
X = rng.standard_normal((20, 4)); y = rng.standard_normal(20)
w = np.linalg.solve(X.T @ X, X.T @ y)
print("normal-equations gradient max abs:", np.max(np.abs(2 * X.T @ (X @ w - y))))

## Visualize the data & results

_Do the analytic gradient identities actually match the numbers? Compare each to a numerical finite-difference gradient._

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def numgrad_vec(f, x, h=1e-6):
    g = np.zeros_like(x)
    for i in range(x.size):
        xp, xm = x.copy(), x.copy(); xp[i] += h; xm[i] -= h
        g[i] = (f(xp) - f(xm)) / (2 * h)
    return g

def numgrad_mat(f, A, h=1e-6):
    G = np.zeros_like(A)
    for idx in np.ndindex(A.shape):
        Ap, Am = A.copy(), A.copy(); Ap[idx] += h; Am[idx] -= h
        G[idx] = (f(Ap) - f(Am)) / (2 * h)
    return G

n = 5
b = rng.standard_normal(n); x = rng.standard_normal(n); A = rng.standard_normal((n, n))
e1 = np.max(np.abs(b - numgrad_vec(lambda v: b @ v, x)))
e2 = np.max(np.abs((A + A.T) @ x - numgrad_vec(lambda v: v @ A @ v, x)))

m = 4
B = rng.standard_normal((m, m)); C = rng.standard_normal((m, m)); M = rng.standard_normal((m, m))
e3 = np.max(np.abs(B.T - numgrad_mat(lambda Z: np.trace(Z @ B), M)))
e4 = np.max(np.abs(C @ M @ B + C.T @ M @ B.T - numgrad_mat(lambda Z: np.trace(Z @ B @ Z.T @ C), M)))
e5 = np.max(np.abs(np.linalg.det(M) * np.linalg.inv(M).T - numgrad_mat(lambda Z: np.linalg.det(Z), M)))

errs = [e1, e2, e3, e4, e5]                       # -> [8.0e-11, 1.2e-10, 4.1e-10, 5.6e-9, 4.6e-10]
labels = ['grad b^T x', 'grad x^T A x', 'grad tr(AB)', 'grad tr(ABA^T C)', 'grad |A|']

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, errs, color=['#4ea1ff', '#7ee787', '#ffb454', '#c89bff', '#ff7b72'])
ax.set_yscale('log'); ax.set_ylabel('max abs error (analytic vs numerical)')
ax.set_title('Each gradient identity matches finite differences')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute $\nabla_x\, x^\top A x$ at $x=\begin{bmatrix}1\\2\end{bmatrix}$ for the symmetric matrix $A=\begin{bmatrix}2&1\\1&3\end{bmatrix}$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check symmetry: $A^\top = \begin{bmatrix}2&1\\1&3\end{bmatrix} = A$, so $A$ is symmetric. — _When $A=A^\top$ the rule simplifies to $2Ax$._
- Apply $2Ax = 2\begin{bmatrix}2&1\\1&3\end{bmatrix}\begin{bmatrix}1\\2\end{bmatrix}$. — _$(A+A^\top)x = 2Ax$ for symmetric $A$._
- Inner product: $Ax = \begin{bmatrix}2\cdot1+1\cdot2\\1\cdot1+3\cdot2\end{bmatrix} = \begin{bmatrix}4\\7\end{bmatrix}$, then double it. — _Matrix times vector first, scalar last._

**Answer:** $\nabla_x\, x^\top A x = 2\begin{bmatrix}4\\7\end{bmatrix} = \begin{bmatrix}8\\14\end{bmatrix}$.

</details>

**Problem 2.** Using the identities, derive the gradient of the ridge-regression objective $\lVert Xw-y\rVert^2 + \lambda w^\top w$ with respect to $w$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The first term's gradient is $2X^\top(Xw-y)$, from the least-squares derivation. — _Quadratic rule on $w^\top X^\top X w$ plus linear rule on $-2y^\top X w$._
- The penalty $\lambda w^\top w$ is a quadratic form with $A=\lambda I$ (symmetric), so its gradient is $2\lambda w$. — _$\nabla_w\, w^\top(\lambda I)w = 2\lambda I w = 2\lambda w$._
- Add the two gradients. — _The gradient of a sum is the sum of the gradients._

**Answer:** $\nabla_w = 2X^\top(Xw-y) + 2\lambda w$. Setting it to zero gives $w=(X^\top X+\lambda I)^{-1}X^\top y$.

</details>

**Problem 3.** For an invertible $2\times2$ matrix $A$ with $|A| = 5$ and $A^{-1}=\begin{bmatrix}0.4&-0.2\\-0.1&0.3\end{bmatrix}$, what is $\nabla_A |A|$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply $\nabla_A|A| = |A|\,(A^{-1})^\top$. — _The determinant identity from the formula block._
- Transpose the inverse: $(A^{-1})^\top = \begin{bmatrix}0.4&-0.1\\-0.2&0.3\end{bmatrix}$. — _Swap the off-diagonal entries._
- Multiply by $|A|=5$. — _Scale every entry by the determinant._

**Answer:** $\nabla_A|A| = 5\begin{bmatrix}0.4&-0.1\\-0.2&0.3\end{bmatrix} = \begin{bmatrix}2&-0.5\\-1&1.5\end{bmatrix}$.

</details>